In [ ]:
# === ノートブック共通の前処理 (llm_math パッケージの読み込み) ===
import sys
from pathlib import Path

# llm_math パッケージの候補パス
_candidates = [
    '.', 'src', '..', '../src',
    '/content/llm-math-book/src',
    '/content/llm-math-book',
    '/workspace/src',
    '/workspace',
]
# 親ディレクトリも候補に追加 (notebooks/ フォルダで実行する場合)
for p in Path.cwd().parents:
    _candidates.append(str(p / 'src'))
    _candidates.append(str(p))

for p in _candidates:
    if p and p not in sys.path and Path(p).exists():
        sys.path.insert(0, p)

# llm_math の import を試行
try:
    from llm_math import viz, bench, data
    _LLM_MATH_OK = True
except ImportError as e:
    _LLM_MATH_OK = False
    print(f"[注意] llm_math パッケージの読み込みエラー: {e}")
    print("  GitHub リポジトリを clone して colab_setup.sh を実行してください。")
# === 前処理ここまで ===


# Ch 16. 位置エンコーディング

> **学習目標**
> -
> - Sinusoidal, Learned, ALiBi, RoPE
> - RoPE 行列 LLM

## 16.1 問題

Attention ** (permutation equivariant)**: 複雑度 出力度 → .

" " vs " " — Attention .

: . .

## 16.2 Sinusoidal Positional Encoding ( )

$$PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d}}\right), \quad PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d}}\right)$$

- $pos$:
- $i$: 次元
- $10000^{-2i/d}$ → エンコード

 : $\mathbf{x} = \mathbf{e} + PE_{pos}$


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def sinusoidal_pe(seq_len, d_model):
    """Sinusoidal Positional Encoding."""
    pe = np.zeros((seq_len, d_model))
    pos = np.arange(seq_len)[:, None]
    div = np.exp(np.arange(0, d_model, 2) * (-np.log(10000) / d_model))
    pe[:, 0::2] = np.sin(pos * div)
    pe[:, 1::2] = np.cos(pos * div)
    return pe

pe = sinusoidal_pe(100, 64)
print(f"PE shape: {pe.shape}")
print(f"PE[0, :8]: {pe[0, :8].round(4)}")
print(f"PE[1, :8]: {pe[1, :8].round(4)}")

# 可視化
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
im = axes[0].imshow(pe, cmap='RdBu', aspect='auto')
plt.colorbar(im, ax=axes[0])
axes[0].set_xlabel('Embedding Dimension')
axes[0].set_ylabel('Position')
axes[0].set_title('Sinusoidal Positional Encoding')

#  Dimension Value
for i in [0, 4, 16, 32]:
    axes[1].plot(pe[:, i], label=f'dim {i}', linewidth=1.5)
axes[1].set_xlabel('Position')
axes[1].set_ylabel('PE Value')
axes[1].set_title('Dimension PE Pattern')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/ch16_sinusoidal_pe.png', dpi=100, bbox_inches='tight')
plt.show()


## 16.3 Learned Positional Embedding

BERT, GPT-2 学習 $\mathbf{E}_{pos} \in \mathbb{R}^{T_{\max} \times d}$ .

: データ 学習.
: 学習 $T_{\max}$ ( ).


In [ ]:
# Learned PE (PyTorch)
import torch
import torch.nn as nn

max_len, d = 512, 64
learned_pe = nn.Embedding(max_len, d)

#    
positions = torch.arange(10).unsqueeze(0)  # (1, 10)
pe_emb = learned_pe(positions)  # (1, 10, d)
print(f"Learned PE: {pe_emb.shape}")
print(f"値 , Trainingの場合 Optimization")


## 16.4 ALiBi — エンコード

**ALiBi** (Press et al., 2022) , Attention :

$$\mathrm{softmax}\left(QK^\top + m \cdot (i - j)\right)$$

- $m$: ( )
- $(i - j)$: 
-

****: 学習 .


In [ ]:
# ALiBi 
def alibi_bias(n_heads, seq_len):
    """ALiBi Bias Matrix: (n_heads, T, T)."""
    # Head Gradient: 2^(-8 * h / n_heads), h=1..n_heads
    slopes = 1.0 / (2 ** (8 * torch.arange(1, n_heads + 1).float() / n_heads))  # (n_heads,)
    # Distance Matrix: (T, T), relative[i, j] = j - i (の場合 , の場合 )
    positions = torch.arange(seq_len)
    relative = positions[None, :] - positions[:, None]  # (T, T)
    # broadcast: (n_heads, 1, 1) * (1, T, T) -> (n_heads, T, T)
    bias = slopes[:, None, None] * relative[None, :, :]
    return bias

n_heads, seq_len = 8, 16
bias = alibi_bias(n_heads, seq_len)
print(f"ALiBi bias: {bias.shape}")

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for h in range(n_heads):
    ax = axes[h // 4, h % 4]
    im = ax.imshow(bias[h].numpy(), cmap='RdBu_r')
    ax.set_title(f'Head {h} (slope={1/(2**(8/n_heads))**(h+1):.4f})')
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.savefig('../figures/ch16_alibi.png', dpi=100, bbox_inches='tight')
plt.show()
print(" Head  Gradient Distance  Decrease Application.")


## 16.5 RoPE (Rotary Position Embedding) — LLaMA

**RoPE** (Su et al., 2021) $m$ / :

$$\mathrm{RoPE}(\mathbf{x}, m) = R_m \mathbf{x}$$

 $R_m$ 行列:

$$R_m = \mathrm{diag}\left(R(m\theta_1), R(m\theta_2), \ldots, R(m\theta_{d/2})\right)$$

$$R(\theta) = \begin{pmatrix} \cos\theta & -\sin\theta \\ \sin\theta & \cos\theta \end{pmatrix}, \quad \theta_i = 10000^{-2i/d}$$

** **:
$$\langle \mathrm{RoPE}(\mathbf{q}, m), \mathrm{RoPE}(\mathbf{k}, n) \rangle = \langle \mathbf{q}, \mathbf{k} \rangle \text{ ( )}$$

→ 内積 ** $m - n$** . エンコード.


In [ ]:
# RoPE 
def rotate_half(x):
    x1, x2 = x[..., :x.shape[-1] // 2], x[..., x.shape[-1] // 2:]
    return torch.cat([-x2, x1], dim=-1)

def apply_rope(x, positions, theta_base=10000.0):
    """x: (B, H, T, d), positions: (T,)."""
    d = x.shape[-1]
    # theta_i = 1 / theta_base^(2i/d)
    freqs = 1.0 / (theta_base ** (torch.arange(0, d, 2).float() / d))  # (d/2,)
    # angles: (T, d/2)
    angles = positions[:, None] * freqs[None, :]
    # cos, sin: (T, d) —  Dimension Iteration
    cos = torch.cos(angles).repeat_interleave(2, dim=-1)  # (T, d)
    sin = torch.sin(angles).repeat_interleave(2, dim=-1)  # (T, d)
    # (1, 1, T, d)
    cos = cos[None, None, :, :]
    sin = sin[None, None, :, :]
    return x * cos + rotate_half(x) * sin

# Test
d = 8
T = 4
x = torch.randn(1, 1, T, d)
positions = torch.arange(T)
x_rope = apply_rope(x, positions)
print(f": {x.shape}")
print(f"RoPE Application : {x_rope.shape}")

#   検証:   
# q at pos 0, k at pos 1 vs q at pos 1, k at pos 2 (  1)
q1 = torch.randn(1, 1, 1, d)
k1 = torch.randn(1, 1, 1, d)
pos_q1 = torch.tensor([0])
pos_k1 = torch.tensor([1])
pos_q2 = torch.tensor([1])
pos_k2 = torch.tensor([2])

dot1 = (apply_rope(q1, pos_q1) * apply_rope(k1, pos_k1)).sum()
dot2 = (apply_rope(q1, pos_q2) * apply_rope(k1, pos_k2)).sum()
print(f"\nRoPE  Distance  Validation:")
print(f"  (q@0, k@1) Inner Product = {dot1.item():.6f}")
print(f"  (q@1, k@2) Inner Product = {dot2.item():.6f}")
print(f"  Difference: {abs(dot1 - dot2).item():.2e} (≈0,   Distance)")


## 16.6 RoPE 可視化

2D RoPE 可視化.


In [ ]:
# 2D RoPE 可視化
fig, ax = plt.subplots(figsize=(8, 8))
d = 2
theta = 1.0  # 

#  ベクトル
v = np.array([2.0, 1.0])
ax.quiver(0, 0, v[0], v[1], angles='xy', scale_units='xy', scale=1,
          color='blue', linewidth=3, label='Original')

#  1, 2, 3, 4 
for m in [1, 2, 3, 4]:
    angle = m * theta
    R = np.array([[np.cos(angle), -np.sin(angle)],
                  [np.sin(angle),  np.cos(angle)]])
    v_rot = R @ v
    ax.quiver(0, 0, v_rot[0], v_rot[1], angles='xy', scale_units='xy', scale=1,
              color='red', alpha=0.5 + 0.1 * m, linewidth=2,
              label=f'pos={m} (Rotation {np.degrees(angle):.0f}°)')

ax.set_xlim(-3, 3); ax.set_ylim(-3, 3)
ax.set_aspect('equal')
ax.axhline(0, color='black', linewidth=0.5)
ax.axvline(0, color='black', linewidth=0.5)
ax.grid(True, alpha=0.3)
ax.legend()
ax.set_title('RoPE: Position m Vector m·θ  Rotation')
plt.tight_layout()
plt.savefig('../figures/ch16_rope_rotation.png', dpi=100, bbox_inches='tight')
plt.show()
print("=> Position    Rotation.  Vector Inner Product Rotation Difference(= Distance) .")


## 16.7

RoPE : 学習 性能 . :

- **Position Interpolation (PI)**: ($m \to m \cdot L_{train}/L_{test}$)
- **NTK-aware**:
- **YaRN**: NTK +

LLaMA-3, Qwen (128K+) .


## 16.8 要点

| | | モデル |
|---|---|---|
| Sinusoidal |  | Transformer |
| Learned | | BERT, GPT-2 |
| ALiBi | | BLOOM |
| RoPE | ( ) | LLaMA, Qwen, Mistral |

## 演習問題

1. Sinusoidal PE 次元 0 次元 32 計算 比較.
2. 学習 PE 100 学習, 0~50 51~99 可視化.
3. ALiBi $m_h = 2^{-8h/H}$ , .
4. RoPE " " 検証.
5. Position Interpolation $m \to m \cdot L_{train}/L_{test}$ RoPE .

> 解答: `solutions/ch16_solutions.ipynb`
